# Geocode

Geocode: Identify station name for xfer stations (please see round 2- final process)

In [36]:
import warnings
warnings.simplefilter("always")

### Read Inputs

In [37]:
## Data Preparation and Summary
import pandas as pd
import numpy as np
import geopandas as gpd

# Set directory
data_dir = "../input/"
interim_dir = '../output/interim/'
output_dir = '../output/'

# df_full = pd.read_excel(data_dir + "od_20240422_sandagca_weighted_combined_draftfinal.xlsx", sheet_name="OD_RESULTS", engine='openpyxl')
df_blue = pd.read_csv(interim_dir + 'survey_blue_line_xfer_routes_on_off_coords.csv')

In [38]:
df_blue.shape

(4396, 24)

In [39]:
sd_rail_stop = gpd.read_file(data_dir + 'sd_rail_stop_gtfs_2023.geojson')

In [40]:
sd_blue_stop = sd_rail_stop[sd_rail_stop['route_name']=='Blue']
sd_blue_stop.shape

(32, 8)

In [41]:
sd_blue_stop.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [42]:
blue_stop_proj = sd_blue_stop.to_crs(epsg=2226)

# (1 mile = 5280 feet)
buffer_distance_feet = 0.15 * 5280
blue_stop_buffer = blue_stop_proj.copy()
blue_stop_buffer["geometry"] = blue_stop_buffer.buffer(buffer_distance_feet)

# Step 4: (Optional) Reproject back to original CRS
blue_stop_buffer = blue_stop_buffer.to_crs(sd_blue_stop.crs)

In [43]:
# blue_stop_buffer.to_file(interim_dir + 'sd_blue_stop_buffer_015_miles.shp')

In [44]:
from shapely.geometry import Point
geometry = [Point(xy) for xy in zip(df_blue['xfer_on_long'], df_blue['xfer_on_lat'])]
gdf_on = gpd.GeoDataFrame(df_blue, geometry=geometry)
gdf_on.set_crs(epsg=4326, inplace=True)

geometry = [Point(xy) for xy in zip(df_blue['xfer_off_long'], df_blue['xfer_off_lat'])]
gdf_off = gpd.GeoDataFrame(df_blue, geometry=geometry)
gdf_off.set_crs(epsg=4326, inplace=True)

,ID,TRIP_FIRST_ROUTE[Code],TRIP_SECOND_ROUTE[Code],TRIP_THIRD_ROUTE[Code],TRIP_FOURTH_ROUTE[Code],ROUTE_DIRECTION[Code],TRIP_NEXT_ROUTE[Code],TRIP_AFTER_ROUTE[Code],TRIP_3RD_ROUTE[Code],TRIP_LAST4TH_RTE[Code],...,STOP_ON [LAT],STOP_ON [LONG],STOP_OFF [ADDR],STOP_OFF [LAT],STOP_OFF [LONG],xfer_on_lat,xfer_on_long,xfer_off_lat,xfer_off_long,geometry
0,31644,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,32.868324,-117.213439,Carlsbad Village Station Stall 4,33.160922,-117.351168,32.711060,-117.153721,32.869204,-117.214011,POINT (-117.21401 32.8692)
1,31758,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,32.868324,-117.213439,N Torrey Pines Rd & Torrey Pines Golf Course,32.905431,-117.243229,32.544568,-117.029552,32.869204,-117.214011,POINT (-117.21401 32.8692)
2,31772,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,32.868324,-117.213439,Camino Del Mar & 24th St,32.968910,-117.267619,32.769914,-117.205062,32.869204,-117.214011,POINT (-117.21401 32.8692)
3,29182,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,32.868324,-117.213439,N Torrey Pines Rd & Torrey Pines Golf Course,32.905431,-117.243229,32.716386,-117.168811,32.869204,-117.214011,POINT (-117.21401 32.8692)
4,29385,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,32.868011,-117.231658,N Torrey Pines Rd & Torrey Pines Golf Course,32.905431,-117.243229,32.721494,-117.169898,32.866862,-117.230462,POINT (-117.23046 32.86686)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4391,71467,MTS_1_Blue,NaN,NaN,NaN,MTS_1_Orange_01,NaN,NaN,NaN,NaN,...,32.705870,-117.153218,City College Station,32.716289,-117.154173,32.685893,-117.124637,32.705870,-117.153218,POINT (-117.15322 32.70587)
4392,20736,NaN,NaN,NaN,NaN,MTS_1_Orange_01,MTS_1_Blue,NaN,NaN,NaN,...,32.708448,-117.095072,12th & Imperial Station,32.705870,-117.153218,32.705870,-117.153218,32.544566,-117.029539,POINT (-117.02954 32.54457)
4393,30499,NaN,NaN,NaN,NaN,MTS_1_Orange_01,MTS_1_Blue,NaN,NaN,NaN,...,32.743296,-117.030783,12th & Imperial Station,32.705870,-117.153218,32.705870,-117.153218,32.638845,-117.099066,POINT (-117.09907 32.63884)
4394,33540,NaN,NaN,NaN,NaN,MTS_1_Orange_01,MTS_1_Blue,NaN,NaN,NaN,...,32.764905,-117.020531,12th & Imperial Station,32.705870,-117.153218,32.705870,-117.153218,32.544566,-117.029539,POINT (-117.02954 32.54457)


In [45]:
joined_on = gpd.sjoin(
    gdf_on,
    blue_stop_buffer,
    how="left",              # keep all target (point) rows
    predicate="intersects"   # match where point intersects buffer
)

joined_off = gpd.sjoin(gdf_off,blue_stop_buffer,how="left",predicate="intersects")

In [46]:
joined_off.shape

(4567, 33)

In [47]:
joined_on.shape

(4541, 33)

In [48]:
# QAQC
# Count unique stop_names per ID
unique_names_count = joined_on.groupby('ID')['stop_name'].nunique()

# IDs with more than 1 unique stop_name
ids_multiple_names = unique_names_count[unique_names_count > 1].index.tolist()

# Get all stop_names for these IDs
conflicting_names = joined_on[joined_on['ID'].isin(ids_multiple_names)].sort_values('ID')

print(conflicting_names[['ID', 'stop_name']].drop_duplicates())
print(conflicting_names[['ID', 'stop_name']].drop_duplicates()['stop_name'].unique())

            ID              stop_name
4274       496  America Plaza Station
4274       496         Santa Fe Depot
187        730  America Plaza Station
187        730         Santa Fe Depot
3228      1229  America Plaza Station
...        ...                    ...
58       74234         Santa Fe Depot
60       74241  America Plaza Station
60       74241         Santa Fe Depot
3512  10054456  America Plaza Station
3512  10054456         Santa Fe Depot

[290 rows x 2 columns]
['America Plaza Station' 'Santa Fe Depot' 'Civic Center Station'
 'Fifth Avenue Station']


In [49]:
# QAQC
# Count unique stop_names per ID
unique_names_count = joined_off.groupby('ID')['stop_name'].nunique()

# IDs with more than 1 unique stop_name
ids_multiple_names = unique_names_count[unique_names_count > 1].index.tolist()

# Get all stop_names for these IDs
conflicting_names = joined_off[joined_off['ID'].isin(ids_multiple_names)].sort_values('ID')

# print(conflicting_names[['ID', 'stop_name']].drop_duplicates())
print(conflicting_names[['ID', 'stop_name']].drop_duplicates()['stop_name'].unique())

['Santa Fe Depot' 'America Plaza Station']


In [50]:
joined_on = joined_on.drop_duplicates('ID')
joined_off = joined_off.drop_duplicates('ID')

In [51]:
_df_blue = df_blue.merge(joined_on[['ID','stop_name']],on='ID',how='left')
_df_blue.shape

(4396, 25)

In [52]:
_df_blue = _df_blue.merge(joined_off[['ID','stop_name']],on='ID',how='left',suffixes=('_on_round2','_off_round2'))
_df_blue.shape

(4396, 26)

In [53]:
_df_blue.columns

Index(['ID', 'TRIP_FIRST_ROUTE[Code]', 'TRIP_SECOND_ROUTE[Code]',
       'TRIP_THIRD_ROUTE[Code]', 'TRIP_FOURTH_ROUTE[Code]',
       'ROUTE_DIRECTION[Code]', 'TRIP_NEXT_ROUTE[Code]',
       'TRIP_AFTER_ROUTE[Code]', 'TRIP_3RD_ROUTE[Code]',
       'TRIP_LAST4TH_RTE[Code]', 'REWEIGHTED_LINKED', 'prev_position',
       'next_position', 'xfer_position', 'STOP_ON [ADDR]', 'STOP_ON [LAT]',
       'STOP_ON [LONG]', 'STOP_OFF [ADDR]', 'STOP_OFF [LAT]',
       'STOP_OFF [LONG]', 'xfer_on_lat', 'xfer_on_long', 'xfer_off_lat',
       'xfer_off_long', 'stop_name_on_round2', 'stop_name_off_round2'],
      dtype='object')

In [54]:
# QAQC
midcoast_stns = [
    'UTC Station',
    'Executive Drive Station',
    'UCSD Health La Jolla Station',
    'UCSD Central Campus Station',
    'VA Medical Center Station',
    'Nobel Drive Station',
    'Balboa Avenue Station',
    'Clairemont Drive Station',
    'Tecolote Road Station'
]

In [55]:
df_blue = _df_blue.copy()
df_blue = df_blue.rename(columns={'stop_name_on_round2':'xfer_on_name','stop_name_off_round2':'xfer_off_name'})

In [56]:
df_blue['xfer_on_midcoast'] = ((df_blue['xfer_on_name'].isin(midcoast_stns)) | (df_blue['xfer_off_name'].isin(midcoast_stns))).astype(int)

In [57]:
df_blue.groupby('xfer_on_midcoast')['REWEIGHTED_LINKED'].sum()

xfer_on_midcoast
0    11108.906728
1     2159.035292
Name: REWEIGHTED_LINKED, dtype: float64

In [58]:
df_blue.to_csv(interim_dir + 'survey_blue_line_xfer_routes_on_off_geocode.csv',index=False)

In [62]:
df_blue.head()

,ID,TRIP_FIRST_ROUTE[Code],TRIP_SECOND_ROUTE[Code],TRIP_THIRD_ROUTE[Code],TRIP_FOURTH_ROUTE[Code],ROUTE_DIRECTION[Code],TRIP_NEXT_ROUTE[Code],TRIP_AFTER_ROUTE[Code],TRIP_3RD_ROUTE[Code],TRIP_LAST4TH_RTE[Code],...,STOP_OFF [ADDR],STOP_OFF [LAT],STOP_OFF [LONG],xfer_on_lat,xfer_on_long,xfer_off_lat,xfer_off_long,xfer_on_name,xfer_off_name,xfer_on_midcoast
0,31644,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,Carlsbad Village Station Stall 4,33.160922,-117.351168,32.711060,-117.153721,32.869204,-117.214011,Park & Market Station,UTC Station,1
1,31758,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,N Torrey Pines Rd & Torrey Pines Golf Course,32.905431,-117.243229,32.544568,-117.029552,32.869204,-117.214011,San Ysidro Station,UTC Station,1
2,31772,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,Camino Del Mar & 24th St,32.968910,-117.267619,32.769914,-117.205062,32.869204,-117.214011,Tecolote Road Station,UTC Station,1
3,29182,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,N Torrey Pines Rd & Torrey Pines Golf Course,32.905431,-117.243229,32.716386,-117.168811,32.869204,-117.214011,America Plaza Station,UTC Station,1
4,29385,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,N Torrey Pines Rd & Torrey Pines Golf Course,32.905431,-117.243229,32.721494,-117.169898,32.866862,-117.230462,County Center/Little Italy Station,Nobel Drive Station,1


# Manual add

In [60]:
# ON

# [add][UTC Station]
# ID 28335: NCTD 101 (survey route) - Blue Line (1st xfer) - Orange Line (2nd xfer)
#     Oceanside Transit Center (survey on) - UTC Transit Center (survey off) - xxx (1st on, near UTC)
#     linked trips: 1.77121492657146


# [add][VA Medical Center Station]
# ID 7572: Blue Line (1st xfer) - MTS 235 (survey route)
#     xxx (1st on, near VA Medical Center) - american plaza (1st off) - Santa Fe Depot Transit Center (survey on) - Rancho Bernardo Transit Station (survey off)
#     linked trips: 3.57683908449606
#     origin place: Medical Services / Hospital (non-work)


# [add][Nobel Drive Station]
# ID 38001: Blue Line (1st xfer) - Orange Line (survey route)
#     xxx (1st on, near nobel drive) - xxx - Courthouse Station (survey on) - Lemon Grove Depot (survey off)
#     linked trips: 3.50992987581353

# [add][Balboa Avenue Station]
# ID 18304: Blue Line (1st xfer) - MTS 7 (survey route)
#     xxx (1st on, near balboa ave) - Civic Center (1st off) - Front St & B St (survey on) - University Av & Chamoune Av (survey off)
#     linked trips: 3.77226768037253

In [61]:
# OFF

# [add][Nobel Drive Station]
# ID 21615: Green Line (survey route) - Blue Line (1st xfer)
#     Gaslamp Quarter Station (survey on) - Old Town Station (off) - Old Town Station (1st on) - xxx (1st off, near nobel drive)
#     linked trips: 38.9009109931936
#     destin place: home
